In [654]:
import pandas as pd

# E-Commerce Customer Intelligence & Business Analytics
## 02 — Data Cleaning

This notebook cleans and prepares the raw Olist e-commerce datasets for exploratory analysis, SQL analysis, and business intelligence.

### Objectives
- Load the raw datasets
- Correct data types
- Investigate and handle missing values
- Handle duplicates where appropriate
- Validate data quality
- Save cleaned datasets separately from the raw data

In [655]:
customers = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_customers_dataset.csv"
)

geolocation = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_geolocation_dataset.csv"
)

order_items = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_order_reviews_dataset.csv"
)

orders = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_orders_dataset.csv"
)

products = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_products_dataset.csv"
)

sellers = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\olist_sellers_dataset.csv"
)

category_translation = pd.read_csv(
    r"C:\Users\ayush\OneDrive\Data analysis project\E-Commerce Customer Intelligence & Business Analytics\Raw Data\product_category_name_translation.csv"
)

In [656]:
tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

## 1. Initial Data Validation

Before modifying any data, we verify that all raw datasets have been loaded successfully.

In [657]:
for name, df in tables.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [658]:
missing_report = pd.DataFrame({
    "missing_values": [df.isna().sum().sum() for df in tables.values()],
    "total_rows": [len(df) for df in tables.values()]
}, index=tables.keys())

missing_report

,missing_values,total_rows
customers,0,99441
geolocation,0,1000163
order_items,11265,112650
payments,9349,103886
reviews,145903,99224
orders,4908,99441
products,7365,32951
sellers,0,3095
category_translation,0,71


In [659]:
for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    
    if len(missing) > 0:
        print(f"\n{name}")
        print(missing)


order_items
price    11265
dtype: int64

payments
payment_value    9349
dtype: int64

reviews
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products
product_category_name         2232
product_name_lenght            610
product_description_lenght     610
product_photos_qty             610
product_weight_g              3297
product_length_cm                2
product_height_cm                2
product_width_cm                 2
dtype: int64


In [660]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [661]:
for column in order_date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

In [662]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

In [663]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

In [664]:
orders[order_date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [665]:
for column in order_date_columns:
    print(column, orders[column].dtype)

order_purchase_timestamp datetime64[us]
order_approved_at datetime64[us]
order_delivered_carrier_date datetime64[us]
order_delivered_customer_date datetime64[us]
order_estimated_delivery_date datetime64[us]


In [666]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

In [667]:
review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

In [668]:
print(order_items["shipping_limit_date"].dtype)

print(reviews["review_creation_date"].dtype)
print(reviews["review_answer_timestamp"].dtype)

datetime64[us]
datetime64[us]
datetime64[us]


In [669]:
orders_missing = pd.DataFrame({
    "missing_count": orders.isna().sum(),
    "missing_percentage": (orders.isna().mean() * 100).round(2)
})

orders_missing = orders_missing[
    orders_missing["missing_count"] > 0
]

orders_missing

,missing_count,missing_percentage
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


In [670]:
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [671]:
orders_missing = pd.DataFrame({
    "missing_count": orders.isna().sum(),
    "missing_percentage": (orders.isna().mean() * 100).round(2)
})

orders_missing = orders_missing[
    orders_missing["missing_count"] > 0
]

orders_missing

,missing_count,missing_percentage
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


In [672]:
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

## 4.1 Investigating Missing Delivery Dates

Orders with missing delivery dates are not automatically treated as errors.

Missing delivery dates can be legitimate for orders that were cancelled, unavailable, processing, or still in transit.

We therefore compare missing delivery dates against `order_status` to identify potential data-quality issues.

In [673]:
delivered_missing_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_missing_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [674]:
delivered_missing_date[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


### Finding

Eight orders are marked as `delivered` but have no
`order_delivered_customer_date`.

These records are treated as data-quality anomalies. The missing
delivery dates will not be artificially filled because the actual
delivery date cannot be reliably determined from the available data.

The original missing values are therefore preserved.

In [675]:
print(
    "Delivered orders with missing delivery date:",
    len(delivered_missing_date)
)

Delivered orders with missing delivery date: 8


### 4.2 Investigating Missing Approval Dates

Approval timestamps are missing for a small number of orders.
We compare these records with their order status to determine
whether the missing values are associated with specific order states.

In [676]:
orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

### 4.3 Investigating Missing Carrier Dates

Carrier delivery timestamps are missing for some orders.
We compare the missing records against order status to determine
whether the absence of a carrier date is expected.

In [677]:
orders[orders["order_delivered_carrier_date"].isna()]["order_status"].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [678]:
orders[orders["order_approved_at"].isna()]["order_status"].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

### Missing Timestamp Findings

The missing timestamps in the `orders` dataset were compared
against `order_status`.

Most missing timestamps occur in order states where the absence
of a timestamp can be logically explained, such as `canceled`,
`unavailable`, `processing`, and `created`.

A small number of records have potentially inconsistent states,
including delivered orders without approval or carrier timestamps.

Because the actual missing timestamps cannot be reliably inferred
from the available data, these values are retained as missing
(`NaT`) rather than being artificially imputed.

In [679]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [680]:
orders["order_status"].isna().sum()

np.int64(0)

## 5. Cleaning Order Items

The `order_items` dataset contains item-level information for
each order, including products, sellers, prices, and freight costs.

We will investigate missing values and validate numerical fields
before deciding how they should be handled.

In [681]:
order_items[order_items["price"].isna()].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
31,001427c0ec99cf8af737bd88e92fd444,1,28b4eced95a52d9c437a4caf9d311b95,77530e9772f57a62c906e1c21538ab82,2018-04-12 19:08:28,NaN,23.35
35,0014ae671de39511f7575066200733b7,1,23365beed316535b4105bd800c46670e,92eb0f42c21942b6552362b9b114707d,2017-05-29 03:15:24,NaN,14.10
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,NaN,17.63
70,0029c5db35ef291ce99d1262012e7daa,1,7614c62b86a81021243e438cfde78ccc,900ba814c251a692506d7834c1218441,2018-09-03 03:50:23,NaN,21.62
87,003324c70b19a16798817b2b3640e721,1,2b939dc9b176d7fa21594d588815d4a4,dbc22125167c298ef99da25668e1011f,2017-05-24 19:22:34,NaN,14.45


In [682]:
order_items[order_items["price"].isna()]["order_id"].nunique()

11075

In [683]:
order_items[order_items["price"].isna()]["product_id"].nunique()

7227

In [684]:
order_items[order_items["price"].isna()].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
31,001427c0ec99cf8af737bd88e92fd444,1,28b4eced95a52d9c437a4caf9d311b95,77530e9772f57a62c906e1c21538ab82,2018-04-12 19:08:28,NaN,23.35
35,0014ae671de39511f7575066200733b7,1,23365beed316535b4105bd800c46670e,92eb0f42c21942b6552362b9b114707d,2017-05-29 03:15:24,NaN,14.10
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,NaN,17.63
70,0029c5db35ef291ce99d1262012e7daa,1,7614c62b86a81021243e438cfde78ccc,900ba814c251a692506d7834c1218441,2018-09-03 03:50:23,NaN,21.62
87,003324c70b19a16798817b2b3640e721,1,2b939dc9b176d7fa21594d588815d4a4,dbc22125167c298ef99da25668e1011f,2017-05-24 19:22:34,NaN,14.45


In [685]:
order_items[order_items["price"].isna()].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
31,001427c0ec99cf8af737bd88e92fd444,1,28b4eced95a52d9c437a4caf9d311b95,77530e9772f57a62c906e1c21538ab82,2018-04-12 19:08:28,NaN,23.35
35,0014ae671de39511f7575066200733b7,1,23365beed316535b4105bd800c46670e,92eb0f42c21942b6552362b9b114707d,2017-05-29 03:15:24,NaN,14.10
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,NaN,17.63
70,0029c5db35ef291ce99d1262012e7daa,1,7614c62b86a81021243e438cfde78ccc,900ba814c251a692506d7834c1218441,2018-09-03 03:50:23,NaN,21.62
87,003324c70b19a16798817b2b3640e721,1,2b939dc9b176d7fa21594d588815d4a4,dbc22125167c298ef99da25668e1011f,2017-05-24 19:22:34,NaN,14.45


In [686]:
order_items[order_items["price"].isna()]["order_id"].nunique()

11075

In [687]:
order_items[order_items["price"].isna()]["product_id"].nunique()

7227

In [688]:
order_items[order_items["price"].isna()].describe(include="all")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,11265,11265.000000,11265,11265,11265,0.0,11265.000000
unique,11075,NaN,7227,1698,NaN,NaN,NaN
top,637617b3ffe9e2f7a2411243829226d0,NaN,422879e10f46682990de24d770e7f83d,6560211a19b47992c3666cc44a7e94c0,NaN,NaN,NaN
freq,4,NaN,52,210,NaN,NaN,NaN
mean,NaN,1.191656,NaN,NaN,2018-01-08 18:31:53.290190,NaN,19.755877
min,NaN,1.000000,NaN,NaN,2016-09-19 23:11:33,NaN,0.000000
25%,NaN,1.000000,NaN,NaN,2017-09-21 12:30:10,NaN,13.000000
50%,NaN,1.000000,NaN,NaN,2018-01-29 15:37:18,NaN,16.280000
75%,NaN,1.000000,NaN,NaN,2018-05-11 12:30:42,NaN,21.120000
max,NaN,20.000000,NaN,NaN,2020-04-09 22:35:08,NaN,306.060000


### Missing Price Investigation

The `order_items` dataset contains 11,265 records with missing
`price` values.

These records still contain valid order, product, seller,
shipping, and freight information. Therefore, removing them would
result in unnecessary data loss.

Since the actual item price cannot be reliably inferred from the
available fields, the missing prices are retained as `NaN`.

For revenue-related analysis, records with missing prices will be
handled explicitly rather than assigning artificial values.

In [689]:
order_items["price"].isna().sum()

np.int64(11265)

### 5.1 Validating Numerical Values

The `price` and `freight_value` columns represent monetary values and
should not contain negative amounts.

Missing prices are retained as `NaN` because they cannot be reliably
reconstructed. We now check for invalid negative values.

In [690]:
print("Negative prices:", (order_items["price"] < 0).sum())
print("Negative freight values:", (order_items["freight_value"] < 0).sum())

Negative prices: 0
Negative freight values: 0


In [691]:
print("Zero prices:", (order_items["price"] == 0).sum())
print("Zero freight values:", (order_items["freight_value"] == 0).sum())

Zero prices: 0
Zero freight values: 383


In [692]:
order_items[["order_item_id", "price", "freight_value"]].dtypes

order_item_id      int64
price            float64
freight_value    float64
dtype: object

### 5.2 Validating Order Item Structure

Each order can contain multiple items. The `order_item_id` identifies
the sequence of items within an order.

We validate the combination of `order_id` and `order_item_id` to
ensure that each item is uniquely represented.

In [693]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [694]:
order_items.loc[
    order_items["freight_value"] == 0,
    ["order_id", "product_id", "price", "freight_value"]
].head(10)

,order_id,product_id,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,53b36df67ebb7c41585e8d54d6772e08,NaN,0.0
258,00a870c6c06346e85335524935c600c0,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,53b36df67ebb7c41585e8d54d6772e08,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,53b36df67ebb7c41585e8d54d6772e08,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0


In [695]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [696]:
order_items.loc[
    order_items["freight_value"] == 0,
    ["order_id", "product_id", "price", "freight_value"]
].head(10)

,order_id,product_id,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,53b36df67ebb7c41585e8d54d6772e08,NaN,0.0
258,00a870c6c06346e85335524935c600c0,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,422879e10f46682990de24d770e7f83d,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,53b36df67ebb7c41585e8d54d6772e08,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,53b36df67ebb7c41585e8d54d6772e08,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,53b36df67ebb7c41585e8d54d6772e08,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,aca2eb7d00ea1a7b8ebd4e68314663af,69.9,0.0


## 6. Cleaning Payments

The `payments` dataset contains payment-level information for each
order, including payment method, payment sequence, installment count,
and payment value.

We will investigate missing payment values and validate the numerical
fields before deciding how the missing records should be handled.

In [697]:
payments_missing = payments[payments["payment_value"].isna()]

payments_missing.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
23,4d680edbaa7d3d9bed69532957368a03,1,credit_card,10,NaN
35,b2bb080b6bc860118a246fd9b6fad6da,1,credit_card,1,NaN
39,00d8d65b666158b633f96054d31af43b,1,boleto,1,NaN
53,1ffb3c1929b16d9c1aec1958e11b3e9b,1,boleto,1,NaN
70,e3e9e3295f4e1512167c232a0aaae8a5,1,voucher,1,NaN


In [698]:
payments_missing["order_id"].nunique()

9281

In [699]:
payments_missing["payment_type"].value_counts()

payment_type
credit_card    6864
boleto         1787
voucher         559
debit_card      139
Name: count, dtype: int64

In [700]:
payments_missing["payment_installments"].value_counts().sort_index()

payment_installments
1     4735
2     1157
3      912
4      604
5      468
6      363
7      156
8      368
9       57
10     493
11       1
12      13
13       1
15       9
16       1
17       1
18       3
20       3
24       4
Name: count, dtype: int64

### Missing Payment Values

The `payments` dataset contains 9,349 records with missing
`payment_value`.

These records still contain valid payment method and installment
information, so they are not treated as duplicate or invalid records.

Because the actual payment amount cannot be reliably reconstructed
from the available payment fields, the missing values are retained
as NaN.

For analyses involving payment amounts, missing values will be
handled explicitly rather than being replaced with artificial values.

In [701]:
print("Negative payment values:", (payments["payment_value"] < 0).sum())
print("Zero payment values:", (payments["payment_value"] == 0).sum())

Negative payment values: 0
Zero payment values: 8


In [702]:
payments[
    ["payment_sequential", "payment_installments", "payment_value"]
].dtypes

payment_sequential        int64
payment_installments      int64
payment_value           float64
dtype: object

In [703]:
payments.duplicated(
    subset=["order_id", "payment_sequential"]
).sum()

np.int64(0)

### Investigating Zero Payment Values

A small number of payment records have a payment value of zero.

Zero-value payments are not automatically treated as invalid because
they may represent legitimate payment records such as promotional or
non-monetary transactions.

We inspect these records before deciding whether any action is required.

In [704]:
payments.loc[
    payments["payment_value"] == 0,
    [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ]
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [705]:
print("Negative payment values:", (payments["payment_value"] < 0).sum())

print(
    "Duplicate payment records:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

Negative payment values: 0
Duplicate payment records: 0


## 7. Cleaning Products

The products dataset contains product-level information such as
category, product dimensions, weight, photographs, and text-length
attributes.

We will identify missing and invalid values, remove records that
cannot provide useful product information, and retain legitimate
missing attributes where appropriate.

In [706]:
products.isna().sum()

product_id                       0
product_category_name         2232
product_name_lenght            610
product_description_lenght     610
product_photos_qty             610
product_weight_g              3297
product_length_cm                2
product_height_cm                2
product_width_cm                 2
dtype: int64

In [707]:
products[products["product_category_name"].isna()][
    ["product_id", "product_category_name"]
].head(10)

,product_id,product_category_name
83,0992c6cba95a13bfa68ea7d5e22d478b,NaN
96,e6a1ff3552ba3305c1cf0a4dde50347f,NaN
105,a41e356c76fab66334f36de622ecbd3a,NaN
128,d8dee61c2034d6d075997acef1870e9b,NaN
145,56139431d72cd51f19eb9f7dae4d1617,NaN
154,46b48281eb6d663ced748f324108c733,NaN
196,68bfdbfe84f03af17733b10362f10020,NaN
197,5fb61f482620cb672f5e586bb132eae9,NaN
210,21db47f6493b06e8e7fc562ec9890e77,NaN
216,c615805546392ec3b8a20c6590c4d400,NaN


In [708]:
products["product_category_name"].isna().sum()

np.int64(2232)

In [709]:
products[
    products["product_name_lenght"].isna() &
    products["product_description_lenght"].isna() &
    products["product_photos_qty"].isna()
].shape

(610, 9)

In [710]:
products[
    products["product_name_lenght"].isna() &
    products["product_description_lenght"].isna() &
    products["product_photos_qty"].isna()
][
    ["product_id", "product_name_lenght",
     "product_description_lenght", "product_photos_qty"]
].head()

,product_id,product_name_lenght,product_description_lenght,product_photos_qty
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN


In [711]:
products[
    products[
        ["product_length_cm", "product_height_cm", "product_width_cm"]
    ].isna().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [712]:
print("Negative weight:", (products["product_weight_g"] < 0).sum())
print("Negative length:", (products["product_length_cm"] < 0).sum())
print("Negative height:", (products["product_height_cm"] < 0).sum())
print("Negative width:", (products["product_width_cm"] < 0).sum())

Negative weight: 0
Negative length: 0
Negative height: 0
Negative width: 0


In [713]:
print("Zero weight:", (products["product_weight_g"] == 0).sum())
print("Zero length:", (products["product_length_cm"] == 0).sum())
print("Zero height:", (products["product_height_cm"] == 0).sum())
print("Zero width:", (products["product_width_cm"] == 0).sum())

Zero weight: 3
Zero length: 0
Zero height: 0
Zero width: 0


In [714]:
products[
    products["product_name_lenght"].isna() &
    products["product_description_lenght"].isna() &
    products["product_photos_qty"].isna()
].shape

(610, 9)

In [715]:
products[
    products[
        ["product_length_cm", "product_height_cm", "product_width_cm"]
    ].isna().any(axis=1)
]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 7.1 Checking Products with No Descriptive Information

Some products have missing product name length, description length, and photo count.

Before removing these records, we check whether they are referenced by order_items.
Products that are linked to transactions should be retained to preserve relationships between datasets.

In [716]:
products_without_details = products[
    products["product_name_lenght"].isna() &
    products["product_description_lenght"].isna() &
    products["product_photos_qty"].isna()
]

used_product_ids = order_items["product_id"].unique()

print(
    "Products without descriptive information:",
    len(products_without_details)
)

print(
    "Of these, used in orders:",
    products_without_details["product_id"].isin(used_product_ids).sum()
)

Products without descriptive information: 610
Of these, used in orders: 610


In [717]:
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

In [718]:
products.loc[
    products["product_weight_g"] == 0,
    "product_weight_g"
] = pd.NA

In [719]:
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

In [720]:
products.isna().sum()

product_id                       0
product_category_name            0
product_name_length            610
product_description_length     610
product_photos_qty             610
product_weight_g              3300
product_length_cm                2
product_height_cm                2
product_width_cm                 2
dtype: int64

### 7.8 Validating Product IDs

Each product should have a unique product_id because it identifies a product in the dataset.

In [721]:
print("Duplicate product IDs:", products["product_id"].duplicated().sum())

Duplicate product IDs: 0


### 7.9 Recovering Missing Product Categories

Some products have missing product_category_name values.

Before dropping these records or replacing the missing values, we check whether the category information can be recovered from the category translation table.

Products whose category cannot be recovered will be retained because they may still be associated with valid orders.

In [722]:
products_with_missing_category = products[
    products["product_category_name"].isna()
]

products_with_missing_category.shape

(0, 9)

### 7.10 Handling Missing Product Attributes

Some products have missing descriptive and physical attributes.

These values cannot be reliably reconstructed from the available product data, so we will not invent replacement values.

We will identify records with missing attributes and determine whether they should be retained or removed based on their usefulness for the analysis.

In [723]:
products_missing_attributes = products[
    products[
        [
            "product_name_length",
            "product_description_length",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

print("Products with missing attributes:", len(products_missing_attributes))

Products with missing attributes: 3867


### 7.11 Checking Missing Attributes for Ordered Products

Some products have missing physical or descriptive attributes.

Since products that appear in orders are important for business analysis, we check whether these missing attributes belong to products that were actually sold.

Products used in orders will be retained even when some attributes are missing.

In [724]:
ordered_product_ids = order_items["product_id"].unique()

products_missing_attributes["product_id"].isin(
    ordered_product_ids
).sum()

np.int64(3867)

Some products used in orders have missing physical or descriptive attributes.

Because these products are associated with actual transactions, they should not be removed from the dataset.

We therefore retain the products and preserve the unavailable attributes as missing values.

The missing attributes will be handled appropriately during specific analyses rather than being artificially imputed.

In [725]:
product_attribute_missing = products[
    [
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].isna().sum()

product_attribute_missing = product_attribute_missing[
    product_attribute_missing > 0
]

product_attribute_missing

product_name_length            610
product_description_length     610
product_photos_qty             610
product_weight_g              3300
product_length_cm                2
product_height_cm                2
product_width_cm                 2
dtype: int64

### 7.13 Checking Missing Physical Dimensions

Only two products have missing length, height, and width values.

We check whether these products are referenced by orders before deciding whether they should be retained.

missing_dimensions = products[
    products[
        [
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].isna().any(axis=1)
]

print("Products with missing dimensions:", len(missing_dimensions))

print(
    "Missing-dimension products used in orders:",
    missing_dimensions["product_id"].isin(ordered_product_ids).sum()
)

### 7.14 Product Cleaning Decision

The products dataset contains some missing descriptive and physical attributes, but all products with missing attributes are associated with valid product records and are used in orders.

Therefore, these product records will be retained.

Missing attributes will remain as NaN when the actual value cannot be reliably determined.

Product categories with missing values have already been represented as "unknown".

Zero product weights were treated as missing because zero is not a meaningful product weight.

No product records will be dropped at this stage.

In [726]:
print("Final product rows:", len(products))
print("Products dropped:", 32951 - len(products))

Final product rows: 32951
Products dropped: 0


## 8. Cleaning Sellers

The sellers dataset contains information about sellers, including their location.

We will investigate missing values, duplicate seller IDs, and validate the seller records before deciding whether any cleaning is required.

In [727]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

### 8.1 Validating Seller IDs

Each seller should have a unique seller_id because it identifies a seller in the dataset.

In [728]:
print(
    "Duplicate seller IDs:",
    sellers["seller_id"].duplicated().sum()
)

Duplicate seller IDs: 0


### 8.2 Checking Seller Usage in Orders

We check whether all sellers in the sellers dataset are referenced by order_items.

This helps validate the relationship between the seller and order_items datasets.

In [729]:
used_seller_ids = order_items["seller_id"].unique()

unused_sellers = sellers[
    ~sellers["seller_id"].isin(used_seller_ids)
]

print("Total sellers:", len(sellers))
print("Sellers used in orders:", sellers["seller_id"].isin(used_seller_ids).sum())
print("Sellers not used in orders:", len(unused_sellers))

Total sellers: 3095
Sellers used in orders: 3095
Sellers not used in orders: 0


### 8.3 Validating Seller Location Fields

Seller location information should be present because seller_city and seller_state are used to analyze seller distribution and regional performance.

In [730]:
print(
    "Missing seller cities:",
    sellers["seller_city"].isna().sum()
)

print(
    "Missing seller states:",
    sellers["seller_state"].isna().sum()
)

Missing seller cities: 0
Missing seller states: 0


### 8.4 Seller Cleaning Decision

The sellers dataset contains no missing values or duplicate seller IDs.

Seller records will therefore be retained without dropping or imputing any values.

In [731]:
print("Final seller rows:", len(sellers))

Final seller rows: 3095


In [732]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [733]:
print(
    "Duplicate seller IDs:",
    sellers["seller_id"].duplicated().sum()
)

Duplicate seller IDs: 0


In [734]:
print("Total sellers:", len(sellers))
print("Sellers used in orders:", sellers["seller_id"].isin(used_seller_ids).sum())
print("Sellers not used in orders:", len(unused_sellers))

Total sellers: 3095
Sellers used in orders: 3095
Sellers not used in orders: 0


## 9. Cleaning Geolocation

The geolocation dataset contains Brazilian ZIP-code prefix locations and their corresponding latitude, longitude, city, and state.

We will investigate duplicate records and validate the location fields before deciding whether duplicate records should be removed.

In [735]:
geolocation.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

### 9.1 Investigating Duplicate Geolocation Records

The geolocation dataset contains duplicate rows. We investigate whether these are exact duplicate records or whether multiple coordinates are associated with the same ZIP-code prefix.

Duplicate records will only be removed after understanding their structure.

In [736]:
print(
    "Exact duplicate rows:",
    geolocation.duplicated().sum()
)

Exact duplicate rows: 261831


### 9.2 Checking ZIP-Code Prefix Structure

A ZIP-code prefix may appear multiple times in the geolocation dataset because different coordinates can be associated with the same prefix.

We therefore check how many unique ZIP-code prefixes exist and whether multiple records exist for the same prefix.

In [737]:
print(
    "Total geolocation rows:",
    len(geolocation)
)

print(
    "Unique ZIP-code prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

print(
    "ZIP-code prefixes with multiple records:",
    (geolocation["geolocation_zip_code_prefix"].value_counts() > 1).sum()
)

Total geolocation rows: 1000163
Unique ZIP-code prefixes: 19015
ZIP-code prefixes with multiple records: 17972


### 9.3 Removing Exact Duplicate Geolocation Records

The duplicate investigation shows that many ZIP-code prefixes have multiple records, so records cannot be reduced to one row per ZIP-code prefix.

However, 261,831 records are exact duplicates across all columns. These records provide no additional information and can be safely removed.

Only exact duplicate rows will be removed.

In [738]:
geolocation = geolocation.drop_duplicates().copy()

print("Rows after removing exact duplicates:", len(geolocation))
print("Remaining duplicate rows:", geolocation.duplicated().sum())

Rows after removing exact duplicates: 738332
Remaining duplicate rows: 0


### 9.4 Validating Geolocation Coordinates

Latitude and longitude values should fall within valid geographic ranges.

Latitude must be between -90 and 90 degrees, while longitude must be between -180 and 180 degrees.

We check for values outside these ranges before deciding whether any records require correction or removal.

In [739]:
print(
    "Invalid latitude values:",
    ((geolocation["geolocation_lat"] < -90) |
     (geolocation["geolocation_lat"] > 90)).sum()
)

print(
    "Invalid longitude values:",
    ((geolocation["geolocation_lng"] < -180) |
     (geolocation["geolocation_lng"] > 180)).sum()
)

Invalid latitude values: 0
Invalid longitude values: 0


### 9.5 Validating Geolocation Data Types

The latitude and longitude columns should contain numeric values because they represent geographic coordinates.

The ZIP-code prefix should also be numeric.

We check the data types before making any changes.

In [740]:
geolocation[
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ]
].dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
dtype: object

### 9.6 Validating Geolocation City and State Fields

The geolocation dataset contains city and state information used for regional analysis.

Although no null values were found, we check for blank or empty text values to ensure that the location fields contain usable information.

In [741]:
print(
    "Blank city values:",
    (geolocation["geolocation_city"].str.strip() == "").sum()
)

print(
    "Blank state values:",
    (geolocation["geolocation_state"].str.strip() == "").sum()
)

Blank city values: 0
Blank state values: 0


### 9.7 Geolocation Cleaning Decision

The geolocation dataset contained 261,831 exact duplicate rows, which were removed because they provided no additional information.

Multiple records for the same ZIP-code prefix were retained because a ZIP-code prefix can legitimately be associated with multiple geographic coordinates.

No missing values, invalid coordinates, or blank city/state fields were found.

Therefore, no additional records or values need to be removed or imputed from the geolocation dataset.

In [742]:
print("Final geolocation rows:", len(geolocation))
print("Remaining duplicate rows:", geolocation.duplicated().sum())

Final geolocation rows: 738332
Remaining duplicate rows: 0


### 10.1 Checking Missing Values

The category translation dataset maps the original Portuguese product category names to their English translations.

We first check for missing values before validating the category mappings.

In [743]:
category_translation.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

### 10.2 Checking Duplicate Category Mappings

Each original product category should have one corresponding English translation.

We check for duplicate category names and duplicate category mappings before using the translation table to enrich the products dataset.

In [744]:
print(
    "Duplicate original categories:",
    category_translation["product_category_name"].duplicated().sum()
)

print(
    "Duplicate English categories:",
    category_translation["product_category_name_english"].duplicated().sum()
)

print(
    "Duplicate translation rows:",
    category_translation.duplicated().sum()
)

Duplicate original categories: 0
Duplicate English categories: 0
Duplicate translation rows: 0


### 10.3 Validating Category Relationship

The translation table should contain the original category names used by the products dataset.

We compare the product categories with the translation table to identify product categories that do not have an available English translation.

In [745]:
product_categories = set(
    products["product_category_name"].dropna().unique()
)

translated_categories = set(
    category_translation["product_category_name"].unique()
)

untranslated_categories = product_categories - translated_categories

print(
    "Unique product categories:",
    len(product_categories)
)

print(
    "Translated categories:",
    len(product_categories & translated_categories)
)

print(
    "Categories without translation:",
    len(untranslated_categories)
)

Unique product categories: 74
Translated categories: 71
Categories without translation: 3


### 10.4 Identifying Untranslated Categories

Three product categories used in the products dataset do not have corresponding entries in the translation table.

We inspect these categories to determine whether they represent legitimate product categories or missing/unknown category values.

In [746]:
print("Categories without translation:")

for category in sorted(untranslated_categories):
    print(category)

Categories without translation:
pc_gamer
portateis_cozinha_e_preparadores_de_alimentos
unknown


### 10.5 Handling Untranslated Categories

Most product categories have an English translation available.

The categories `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` do not have corresponding entries in the translation table. These are valid category values rather than missing values.

The `unknown` category represents products whose original category was missing and was previously handled during product cleaning.

These categories will be retained rather than removed.

In [747]:
print(
    "Categories retained without translation:",
    len(untranslated_categories)
)

print(
    "Products in untranslated categories:",
    products["product_category_name"].isin(
        untranslated_categories
    ).sum()
)

Categories retained without translation: 3
Products in untranslated categories: 2244


### 10.6 Applying Category Translations

The category translation table provides English names for 71 product categories.

We will merge these translations into the products dataset using the original product category name.

Products without an available translation will remain without an English category rather than being removed.

In [748]:
products = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

print("Products rows after merge:", len(products))
print("Products columns after merge:", len(products.columns))

Products rows after merge: 32951
Products columns after merge: 10


### 10.7 Validating the Category Translation Merge

After merging the English category names into the products dataset, we verify that the merge did not create duplicate product records and identify products that still have no English category translation.

The products dataset should retain its original number of rows.

In [749]:
print(
    "Products with missing English category:",
    products["product_category_name_english"].isna().sum()
)

print(
    "Duplicate product IDs after merge:",
    products["product_id"].duplicated().sum()
)

print(
    "Final product rows:",
    len(products)
)

Products with missing English category: 2244
Duplicate product IDs after merge: 0
Final product rows: 32951


### 10.8 Category Translation Cleaning Decision

The category translation dataset contains 71 unique category mappings with no missing values or duplicate mappings.

The translation table was successfully merged with the products dataset without increasing the number of product records or creating duplicate product IDs.

Most product categories now have English translations. The categories `pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`, and `unknown` do not have corresponding English translations and are therefore retained without an English category.

No category translation records or product records need to be dropped.

In [750]:
print("Final category translation rows:", len(category_translation))
print("Final product rows:", len(products))
print(
    "Products with missing English category:",
    products["product_category_name_english"].isna().sum()
)
print(
    "Duplicate product IDs:",
    products["product_id"].duplicated().sum()
)

Final category translation rows: 71
Final product rows: 32951
Products with missing English category: 2244
Duplicate product IDs: 0


## 11. Cleaning Reviews

The reviews dataset contains customer review scores, review comments, and review timestamps.

We will investigate missing review text, validate review scores and dates, and check for duplicate review records before deciding whether any records or values need to be removed or retained.

In [751]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

### 11.1 Validating Review Scores

Review scores should be integer values from 1 to 5.

We check whether any review scores fall outside this expected range.

In [752]:
print(
    "Invalid review scores:",
    ((reviews["review_score"] < 1) |
     (reviews["review_score"] > 5)).sum()
)

Invalid review scores: 0


### 11.2 Checking Review Score Distribution

We examine the distribution of review scores to understand whether the dataset contains valid scores across the expected range.

In [753]:
print(
    reviews["review_score"].value_counts().sort_index()
)

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


### 11.3 Investigating Missing Review Text

Review comments are optional customer-provided text, so missing review titles or messages do not necessarily indicate invalid records.

We check whether reviews with missing text still contain valid review scores and order information before deciding whether any records should be removed.

In [754]:
reviews[
    reviews["review_comment_title"].isna() &
    reviews["review_comment_message"].isna()
].shape

(56518, 7)

### 11.4 Validating Review Order Relationships

Reviews should be associated with valid orders in the orders dataset.

We check whether any reviews reference an order_id that does not exist in the orders dataset.

In [755]:
invalid_review_orders = reviews[
    ~reviews["order_id"].isin(orders["order_id"])
]

print(
    "Reviews with invalid order IDs:",
    len(invalid_review_orders)
)

Reviews with invalid order IDs: 0


### 11.5 Checking Duplicate Review Records

Each review should have a unique review_id.

We check for duplicate review IDs to ensure that the same review is not represented multiple times.

In [756]:
print(
    "Duplicate review IDs:",
    reviews["review_id"].duplicated().sum()
)

Duplicate review IDs: 814


### 11.6 Investigating Duplicate Review IDs

The reviews dataset contains 814 duplicate review_id values.

Before removing these records, we check whether the duplicate review IDs represent exact duplicate rows or whether the same review_id is associated with different review information.

We will only remove records after understanding the structure of these duplicates.

In [757]:
duplicate_review_ids = reviews[
    reviews["review_id"].duplicated(keep=False)
]

print(
    "Rows involved in duplicate review IDs:",
    len(duplicate_review_ids)
)

print(
    "Unique duplicate review IDs:",
    duplicate_review_ids["review_id"].nunique()
)

Rows involved in duplicate review IDs: 1603
Unique duplicate review IDs: 789


### 11.7 Investigating Duplicate Review Records

Some review IDs appear more than once. However, duplicate review IDs do not necessarily mean that the entire review record is duplicated.

We compare the duplicate records to determine whether they are exact duplicates or whether the same review_id is associated with different review information.

This helps us decide whether any records should be removed.

In [758]:
print(
    "Exact duplicate rows among duplicate review IDs:",
    duplicate_review_ids.duplicated().sum()
)

Exact duplicate rows among duplicate review IDs: 0


### 11.8 Investigating Duplicate Review IDs Further

The duplicate review IDs are not exact duplicate rows.

We therefore inspect whether the same review_id is associated with different orders, review scores, comments, or timestamps.

This will help determine whether the duplicate review IDs represent meaningful repeated records or data-quality inconsistencies.

In [759]:
duplicate_review_ids.groupby("review_id").agg(
    unique_orders=("order_id", "nunique"),
    unique_scores=("review_score", "nunique"),
    unique_titles=("review_comment_title", "nunique"),
    unique_messages=("review_comment_message", "nunique"),
    unique_creation_dates=("review_creation_date", "nunique"),
    unique_answer_timestamps=("review_answer_timestamp", "nunique")
).sort_values(
    "unique_orders",
    ascending=False
).head(10)

,unique_orders,unique_scores,unique_titles,unique_messages,unique_creation_dates,unique_answer_timestamps
review_id,,,,,,
4219a80ab469e3fc9901437b73da3f75,3,1,0,1,1,1
7b606b0d57b078384f0b58eac1d41d78,3,1,0,0,1,1
0c76e7a547a531e7bf9f0b99cba071c1,3,1,0,0,1,1
69a1068c3128a14994e3e422e4539e04,3,1,0,0,1,1
308316408775d1600dad81bd3184556d,3,1,0,1,1,1
44e9f871226d8a130de3fc39dfbdf0c5,3,1,0,0,1,1
2172867fd5b1a55f98fe4608e1547b4b,3,1,0,1,1,1
9e25d6e3025e9b9a0fc7f03588d33e2b,3,1,1,1,1,1
38821b5c496b678cf91acc34892805ad,3,1,0,0,1,1


### 11.9 Checking Review ID and Order Relationship

Some review IDs are associated with multiple orders.

Before removing any records, we check whether each order has multiple reviews and whether the combination of order_id and review_id provides a more appropriate representation of the review records.

We will use the relationship between review_id and order_id to determine whether any records should be removed.

In [760]:
print(
    "Duplicate order IDs:",
    reviews["order_id"].duplicated().sum()
)

print(
    "Orders with multiple review records:",
    (reviews["order_id"].value_counts() > 1).sum()
)

Duplicate order IDs: 551
Orders with multiple review records: 547


### 11.10 Checking Review Records Per Order

Some orders have multiple review records.

Before removing any records, we inspect how many reviews are associated with each order and determine whether these records contain different review scores or information.

Records will only be removed if they are confirmed to be redundant duplicates.

In [761]:
multiple_review_orders = reviews[
    reviews["order_id"].duplicated(keep=False)
]

multiple_review_orders.groupby("order_id").agg(
    review_count=("review_id", "count"),
    unique_review_ids=("review_id", "nunique"),
    unique_scores=("review_score", "nunique"),
    unique_creation_dates=("review_creation_date", "nunique")
).sort_values(
    "review_count",
    ascending=False
).head(10)

,review_count,unique_review_ids,unique_scores,unique_creation_dates
order_id,,,,
03c939fd7fd3b38f8485a0f95798f1f6,3,3,2,3
8e17072ec97ce29f0e1f111e598b0c85,3,3,1,2
c88b1d1b157a9999ce368f218a407141,3,3,2,2
df56136b8031ecd28e200bb18e6ddb2e,3,3,1,3
02355020fd0a40a0d56df9f6ff060413,2,2,2,2
029863af4b968de1e5d6a82782e662f5,2,2,2,2
02e0b68852217f5715fb9cc885829454,2,2,1,2
02e723e8edb4a123d414f56cc9c4665e,2,2,1,2
03515a836bb855b03f7df9dee520a8fc,2,2,1,2


### 11.11 Checking for Redundant Review Records

Multiple review records can legitimately be associated with the same order.

We therefore check whether any records have the same order, review score, review text, and timestamps.

Only records that are confirmed to contain the same review information will be considered redundant and removed.

In [762]:
review_duplicate_columns = [
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

print(
    "Redundant review records:",
    reviews.duplicated(
        subset=review_duplicate_columns
    ).sum()
)

Redundant review records: 0


### 11.12 Validating Review Dates

Review creation and answer timestamps should contain valid dates.

We check whether any review creation dates occur after the review answer timestamps and whether any invalid date relationships exist.

No dates will be modified unless an invalid relationship is identified.

In [763]:
print(
    "Reviews with creation date after answer timestamp:",
    (
        reviews["review_creation_date"]
        > reviews["review_answer_timestamp"]
    ).sum()
)

Reviews with creation date after answer timestamp: 0


### 11.13 Review Cleaning Decision

The reviews dataset contains missing review titles and messages, but review text is optional and these records still contain valid review information.

All review scores are within the valid range of 1 to 5, and all reviews reference valid orders.

Although duplicate review IDs exist, the duplicate records are not exact duplicates, and no redundant review records were identified based on order, score, text, and timestamps.

Review creation dates and answer timestamps also have valid chronological relationships.

Therefore, no review records will be dropped, and missing review text will remain as NaN.

The reviews dataset is retained without further modification.

In [764]:
print("Final review rows:", len(reviews))
print("Missing review titles:", reviews["review_comment_title"].isna().sum())
print("Missing review messages:", reviews["review_comment_message"].isna().sum())
print("Redundant review records:", reviews.duplicated(
    subset=review_duplicate_columns
).sum())

Final review rows: 99224
Missing review titles: 87656
Missing review messages: 58247
Redundant review records: 0


In [765]:
payments.isna().sum()

order_id                   0
payment_sequential         0
payment_type               0
payment_installments       0
payment_value           9349
dtype: int64

### 12.2 Investigating Missing Payment Values

The payment_value column contains 9,349 missing values.

Before deciding whether these records should be removed or imputed, we check whether the affected payment records are associated with valid orders and whether other payment information is available.

Missing payment values will only be removed or imputed if there is sufficient evidence that they represent invalid records.

In [766]:
missing_payment_values = payments[
    payments["payment_value"].isna()
]

print(
    "Payments with missing payment values:",
    len(missing_payment_values)
)

print(
    "Missing-value payments with valid orders:",
    missing_payment_values["order_id"].isin(
        orders["order_id"]
    ).sum()
)

print(
    "Missing-value payments with payment type:",
    missing_payment_values["payment_type"].notna().sum()
)

print(
    "Missing-value payments with installments:",
    missing_payment_values["payment_installments"].notna().sum()
)

Payments with missing payment values: 9349
Missing-value payments with valid orders: 9349
Missing-value payments with payment type: 9349
Missing-value payments with installments: 9349


12.3 Investigating Missing Payment Values by Payment Type

Payment values should normally be available for completed payment records. We check whether missing payment values are concentrated in specific payment types before deciding how they should be handled.

In [767]:
print(
    missing_payment_values["payment_type"].value_counts()
)

payment_type
credit_card    6864
boleto         1787
voucher         559
debit_card      139
Name: count, dtype: int64


In [768]:
print("Zero payment values:", (payments["payment_value"] == 0).sum())
print("Negative payment values:", (payments["payment_value"] < 0).sum())

Zero payment values: 8
Negative payment values: 0


In [769]:
zero_payment_values = payments[
    payments["payment_value"] == 0
]

zero_payment_values

,order_id,payment_sequential,payment_type,payment_installments,payment_value
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [770]:
print(
    payments["payment_type"].value_counts(dropna=False)
)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [771]:
undefined_payments = payments[
    payments["payment_type"] == "not_defined"
]

undefined_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [772]:
undefined_payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

### 12.3 Payment Type Cleaning Decision

The payment dataset contains three records with the payment type `not_defined`.

These records contain valid order IDs, payment installments, and payment values, with no missing values. Although the payment type is undefined, there is insufficient evidence to determine a correct replacement value.

Therefore, the records will be retained and the `not_defined` payment type will remain unchanged.

In [773]:
print(
    "Undefined payment records:",
    (payments["payment_type"] == "not_defined").sum()
)

print(
    "Final payment rows:",
    len(payments)
)

Undefined payment records: 3
Final payment rows: 103886


In [774]:
print(
    payments["payment_installments"].value_counts().sort_index()
)

print(
    "Zero installments:",
    (payments["payment_installments"] == 0).sum()
)

print(
    "Negative installments:",
    (payments["payment_installments"] < 0).sum()
)

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64
Zero installments: 2
Negative installments: 0


In [775]:
zero_installments = payments[
    payments["payment_installments"] == 0
]

zero_installments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


### 12.4 Removing Invalid Payment Installments

Two payment records contain zero installments despite having valid credit card payments with non-zero payment values.

Since a payment cannot meaningfully have zero installments, these two records are considered invalid and will be removed.

In [776]:
payments = payments[
    payments["payment_installments"] > 0
].copy()

print("Rows after removing invalid installments:", len(payments))
print(
    "Remaining zero installments:",
    (payments["payment_installments"] == 0).sum()
)

Rows after removing invalid installments: 103884
Remaining zero installments: 0


In [777]:
missing_payment_values = payments[
    payments["payment_value"].isna()
]

print(
    "Missing payment records:",
    len(missing_payment_values)
)

print(
    "Missing payment records with multiple payments in same order:",
    missing_payment_values["order_id"].duplicated(keep=False).sum()
)

Missing payment records: 9349
Missing payment records with multiple payments in same order: 125


In [778]:
missing_payment_orders = payments[
    payments["order_id"].isin(
        missing_payment_values.loc[
            missing_payment_values["order_id"].duplicated(keep=False),
            "order_id"
        ]
    )
].sort_values("order_id")

missing_payment_orders.head(20)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
86558,08fa8b49ff198d1332df4668087150ed,3,voucher,1,NaN
103499,08fa8b49ff198d1332df4668087150ed,2,voucher,1,50.00
45637,08fa8b49ff198d1332df4668087150ed,1,credit_card,1,NaN
55542,15226cfb3e9f4ad163d09f493688204b,1,voucher,1,NaN
42619,15226cfb3e9f4ad163d09f493688204b,3,voucher,1,NaN
62568,15226cfb3e9f4ad163d09f493688204b,2,voucher,1,36.80
79187,1a292836ced1aa222199c34c883f404e,2,voucher,1,NaN
15502,1a292836ced1aa222199c34c883f404e,1,credit_card,1,NaN
55519,1b16849c735fb603f647c0e1dbf503c7,2,voucher,1,13.87
6444,1b16849c735fb603f647c0e1dbf503c7,1,credit_card,1,3.96


In [ ]:
missing_payment_orders_summary = (
    payments[payments["payment_value"].isna()]
    .groupby("order_id")
    .agg(
        payment_records=("payment_value", "size"),
        missing_values=("payment_value", "count"),
        valid_values=("payment_value", lambda x: x.notna().sum())
    )
)

print(
    "Orders with missing payment values:",
    len(missing_payment_orders_summary)
)

print(
    "Orders with both missing and valid payment values:",
    (
        payments.groupby("order_id")["payment_value"]
        .apply(lambda x: x.isna().any() and x.notna().any())
        .sum()
    )
)

In [ ]:
print(
    "Exact duplicate payment records:",
    payments.duplicated().sum()
)

In [ ]:
print(
    "Duplicate order + payment sequence:",
    payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

### 12.5 Final Payment Cleaning Decision

The payment dataset contains missing payment values, but these records have valid order IDs and other payment information, so the missing values are retained as NaN.

Two records had zero payment installments and were removed because installment counts should be greater than zero.

No exact duplicate payment records or duplicate order-payment sequence combinations were found.

The three `not_defined` payment types are retained because they represent valid payment records and are too few to justify removal.

In [ ]:
print("Final payment rows:", len(payments))
print("Missing payment values:", payments["payment_value"].isna().sum())
print("Zero installments:", (payments["payment_installments"] == 0).sum())
print("Negative installments:", (payments["payment_installments"] < 0).sum())
print("Exact duplicate rows:", payments.duplicated().sum())

### 12.5 Final Payment Cleaning Decision

The `payments` dataset was reviewed for missing values, invalid installment counts, undefined payment types, and duplicate records.

- Missing `payment_value` values were retained as `NaN` because the affected records are associated with valid orders and contain other payment information.
- Two records with zero payment installments were removed.
- No negative installment values were found.
- No exact duplicate records or duplicate `order_id` + `payment_sequential` combinations were found.
- The three `not_defined` payment types were retained due to their very small number.

The cleaned `payments` dataset contains **103,884 records**.

In [ ]:
print("Final payment rows:", len(payments))
print("Missing payment values:", payments["payment_value"].isna().sum())
print("Zero installments:", (payments["payment_installments"] == 0).sum())
print("Negative installments:", (payments["payment_installments"] < 0).sum())
print("Exact duplicate rows:", payments.duplicated().sum())

In [ ]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isna().sum())

In [ ]:
missing_prices = order_items[
    order_items["price"].isna()
]

print("Missing price records:", len(missing_prices))
print("Valid orders:", missing_prices["order_id"].isin(orders["order_id"]).sum())
print("Valid products:", missing_prices["product_id"].isin(products["product_id"]).sum())
print("Valid sellers:", missing_prices["seller_id"].isin(sellers["seller_id"]).sum())

In [ ]:
missing_price_items = order_items[
    order_items["price"].isna()
].copy()

product_price_summary = (
    order_items[order_items["price"].notna()]
    .groupby("product_id")["price"]
    .agg(
        valid_price_count="count",
        min_price="min",
        max_price="max"
    )
)

missing_price_analysis = missing_price_items[
    ["order_id", "product_id", "seller_id", "freight_value"]
].merge(
    product_price_summary,
    on="product_id",
    how="left"
)

missing_price_analysis.head(20)

In [ ]:
print(
    "Missing-price records with known product prices:",
    missing_price_analysis["valid_price_count"].notna().sum()
)

print(
    "Missing-price records with no known product prices:",
    missing_price_analysis["valid_price_count"].isna().sum()
)

In [ ]:
print("Total order_items:", len(order_items))
print("Missing prices:", order_items["price"].isna().sum())
print("Valid prices:", order_items["price"].notna().sum())
print("Negative prices:", (order_items["price"] < 0).sum())
print("Zero prices:", (order_items["price"] == 0).sum())

In [ ]:
print("Exact duplicate rows:", order_items.duplicated().sum())

print(
    "Duplicate order + product + seller combinations:",
    order_items.duplicated(
        subset=["order_id", "product_id", "seller_id"]
    ).sum()
)

In [ ]:
order_items.loc[
    order_items["price"].isna() &
    order_items["product_id"].isin(
        missing_price_analysis.loc[
            missing_price_analysis["min_price"].notna(),
            "product_id"
        ]
    ),
    "price"
] = order_items.loc[
    order_items["price"].isna() &
    order_items["product_id"].isin(
        missing_price_analysis.loc[
            missing_price_analysis["min_price"].notna(),
            "product_id"
        ]
    ),
    "product_id"
].map(
    missing_price_analysis.drop_duplicates("product_id")
    .set_index("product_id")["min_price"]
)

In [ ]:
print("Missing prices after imputation:", order_items["price"].isna().sum())

In [ ]:
remaining_missing_prices = order_items[
    order_items["price"].isna()
]

print("Remaining missing prices:", len(remaining_missing_prices))
print("Unique products:", remaining_missing_prices["product_id"].nunique())

In [ ]:
remaining_missing_prices.head(10)

In [ ]:
print("Remaining missing prices:", order_items["price"].isna().sum())

In [ ]:
order_items = order_items.dropna(subset=["price"]).copy()

print("Rows after removing unrecoverable prices:", len(order_items))
print("Remaining missing prices:", order_items["price"].isna().sum())

### 11.5 Final `order_items` Cleaning Decision

Missing `price` values were handled using product-level historical prices where available. The remaining 1,953 records had no reliable historical price information and were removed.

No negative or zero prices were found. No exact duplicate rows were found.

The cleaned `order_items` dataset contains 110,697 records with no missing prices.

In [ ]:
print("Final order_items rows:", len(order_items))
print("Missing prices:", order_items["price"].isna().sum())
print("Negative prices:", (order_items["price"] < 0).sum())
print("Zero prices:", (order_items["price"] == 0).sum())
print("Exact duplicate rows:", order_items.duplicated().sum())

In [ ]:
reviews.isna().sum()

In [ ]:
print("Review score distribution:")
print(reviews["review_score"].value_counts().sort_index())

print("\nInvalid review scores:")
print(
    reviews[
        ~reviews["review_score"].isin([1, 2, 3, 4, 5])
    ][["order_id", "review_score"]]
)

In [ ]:
print("Minimum review score:", reviews["review_score"].min())
print("Maximum review score:", reviews["review_score"].max())
print("Missing review scores:", reviews["review_score"].isna().sum())

In [ ]:
print("Exact duplicate rows:", reviews.duplicated().sum())

print(
    "Duplicate review_id:",
    reviews["review_id"].duplicated().sum()
)

print(
    "Duplicate order_id:",
    reviews["order_id"].duplicated().sum()
)

In [ ]:
print(
    "Reviews with valid orders:",
    reviews["order_id"].isin(orders["order_id"]).sum()
)

print(
    "Reviews with invalid orders:",
    (~reviews["order_id"].isin(orders["order_id"])).sum()
)

In [ ]:
reviews = reviews.drop(
    columns=[
        "review_comment_title",
        "review_comment_message"
    ]
).copy()

print("Columns after removing unnecessary fields:")
print(reviews.columns)

In [ ]:
print(reviews.isna().sum())

In [ ]:
duplicate_review_ids = reviews[
    reviews["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicate_review_ids.head(20)

In [ ]:
print(
    "Duplicate review IDs:",
    reviews["review_id"].duplicated().sum()
)

print(
    "Exact duplicate rows:",
    reviews.duplicated().sum()
)

In [ ]:
print(
    "Reviews with valid orders:",
    reviews["order_id"].isin(orders["order_id"]).sum()
)

print(
    "Reviews with invalid orders:",
    (~reviews["order_id"].isin(orders["order_id"])).sum()
)

In [ ]:
print("Final review rows:", len(reviews))
print("Missing values:")
print(reviews.isna().sum())

print("Invalid review scores:",
      (~reviews["review_score"].isin([1, 2, 3, 4, 5])).sum())

print("Exact duplicate rows:",
      reviews.duplicated().sum())

In [ ]:
orders.isna().sum()

In [ ]:
print(orders["order_status"].value_counts())

In [ ]:
print(
    orders.groupby("order_status")[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].apply(lambda x: x.isna().sum())
)

In [ ]:
print("Unique order statuses:")
print(orders["order_status"].unique())

In [ ]:
print(
    "Missing order status:",
    orders["order_status"].isna().sum()
)

In [ ]:
print(
    "Approved before purchase:",
    (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Carrier date before purchase:",
    (
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before purchase:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

In [ ]:
print(
    "Customer delivery before carrier:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum()
)

In [ ]:
print("Exact duplicate rows:", orders.duplicated().sum())

print(
    "Duplicate order IDs:",
    orders["order_id"].duplicated().sum()
)

In [ ]:
print(
    "Orders with invalid customers:",
    (~orders["customer_id"].isin(customers["customer_id"])).sum()
)

In [ ]:
print(
    "Orders with missing order IDs:",
    orders["order_id"].isna().sum()
)

print(
    "Orders with missing customer IDs:",
    orders["customer_id"].isna().sum()
)

In [ ]:
print(orders["order_status"].value_counts())

In [ ]:
print(
    orders.groupby("order_status")[
        [
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].apply(lambda x: x.isna().sum())
)

In [ ]:
invalid_carrier_dates = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
]

print("Invalid carrier dates:", len(invalid_carrier_dates))

invalid_carrier_dates[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(20)

In [ ]:
print(
    invalid_carrier_dates["order_status"].value_counts()
)

In [ ]:
invalid_carrier_dates = invalid_carrier_dates.copy()

invalid_carrier_dates["carrier_before_purchase_hours"] = (
    invalid_carrier_dates["order_purchase_timestamp"]
    - invalid_carrier_dates["order_delivered_carrier_date"]
).dt.total_seconds() / 3600

print(
    invalid_carrier_dates[
        "carrier_before_purchase_hours"
    ].describe()
)

In [ ]:
invalid_delivery_dates = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
]

print("Invalid delivery sequence:", len(invalid_delivery_dates))

invalid_delivery_dates[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
]

In [ ]:
delivered_missing_dates = orders[
    (orders["order_status"] == "delivered") &
    (
        orders[
            [
                "order_approved_at",
                "order_delivered_carrier_date",
                "order_delivered_customer_date"
            ]
        ].isna().any(axis=1)
    )
]

print("Delivered orders with missing dates:",
      len(delivered_missing_dates))

delivered_missing_dates

In [ ]:
invalid_carrier_dates["order_status"].value_counts()

In [ ]:
invalid_delivery_dates

In [ ]:
delivered_missing_dates

In [ ]:
print(
    invalid_carrier_dates[
        "carrier_before_purchase_hours"
    ].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
)

In [ ]:
print(
    invalid_carrier_dates[
        "carrier_before_purchase_hours"
    ].sort_values(ascending=False).head(20)
)

In [ ]:
extreme_carrier_dates = invalid_carrier_dates[
    invalid_carrier_dates["carrier_before_purchase_hours"] > 24
]

print(extreme_carrier_dates[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "carrier_before_purchase_hours"
    ]
])

In [ ]:
orders.loc[
    orders["order_id"].isin(extreme_carrier_dates["order_id"]),
    "order_delivered_carrier_date"
] = pd.NaT

In [ ]:
print(
    orders["order_delivered_carrier_date"].isna().sum()
)

In [ ]:
invalid_delivery_dates = invalid_delivery_dates.copy()

invalid_delivery_dates["delivery_before_carrier_hours"] = (
    invalid_delivery_dates["order_delivered_carrier_date"]
    - invalid_delivery_dates["order_delivered_customer_date"]
).dt.total_seconds() / 3600

print(
    invalid_delivery_dates[
        "delivery_before_carrier_hours"
    ].describe()
)

In [ ]:
print(
    invalid_delivery_dates[
        [
            "order_id",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "delivery_before_carrier_hours"
        ]
    ].sort_values(
        "delivery_before_carrier_hours",
        ascending=False
    )
)

In [ ]:
orders.loc[
    orders["order_id"].isin(invalid_delivery_dates["order_id"]),
    "order_delivered_carrier_date"
] = pd.NaT

In [ ]:
print(
    orders["order_delivered_carrier_date"].isna().sum()
)

In [ ]:
print(
    "Delivery before carrier:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum()
)

In [ ]:
print(orders.isna().sum())

In [ ]:
print(orders.info())

In [ ]:
orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
print(orders["delivery_days"].describe())

In [ ]:
print(orders[["order_status", "delivery_days"]].head(10))

In [ ]:
orders["delivery_days"]

In [ ]:
print(
    orders[
        (orders["order_status"] == "delivered") &
        (orders["delivery_days"].isna())
    ][
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ]
)

In [ ]:
orders["order_delivered_customer_date"]

In [ ]:
orders["approval_delay_hours"] = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

In [ ]:
print(orders["approval_delay_hours"].describe())

In [ ]:
print(
    orders[
        [
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "approval_delay_hours"
        ]
    ].head(10)
)

In [ ]:
orders["approval_delay_hours"] = (
    orders["order_approved_at"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

In [ ]:
extreme_approval = orders[
    orders["approval_delay_hours"] > 24
]

print("Orders approved after 24 hours:", len(extreme_approval))

print(
    extreme_approval[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "approval_delay_hours"
        ]
    ]
    .sort_values(
        "approval_delay_hours",
        ascending=False
    )
    .head(20)
)

In [ ]:
print(
    orders["approval_delay_hours"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995]
    )
)

In [ ]:
print(
    pd.cut(
        orders["approval_delay_hours"],
        bins=[0, 1, 6, 12, 24, 48, 168, float("inf")],
        labels=[
            "<1 hour",
            "1-6 hours",
            "6-12 hours",
            "12-24 hours",
            "1-2 days",
            "2-7 days",
            ">7 days"
        ]
    ).value_counts().sort_index()
)

In [ ]:
extreme_approval = orders[
    orders["approval_delay_hours"] > 168
]

print(
    extreme_approval[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "approval_delay_hours"
        ]
    ]
    .sort_values(
        "approval_delay_hours",
        ascending=False
    )
)

In [ ]:
print(
    extreme_approval["order_status"].value_counts()
)

In [ ]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders.loc[
    orders["order_status"] != "delivered",
    "delivery_days"
] = pd.NA

In [ ]:
print(orders[["order_status", "delivery_days"]].head(10))

In [ ]:
print(orders["delivery_days"].describe())

In [ ]:
print(
    orders[
        orders["delivery_days"] < 0
    ][
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ]
)

In [ ]:
print(
    orders[
        orders["delivery_days"] > 100
    ][
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ]
    .sort_values("delivery_days", ascending=False)
)

In [ ]:
orders["delivery_difference_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
print(
    orders["delivery_difference_days"].describe()
)

In [ ]:
print(
    orders[
        orders["order_status"].eq("delivered") &
        orders["delivery_difference_days"].isna()
    ][
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_difference_days"
        ]
    ]
)

In [ ]:
print(
    orders[
        orders["delivery_difference_days"] > 30
    ][
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_difference_days"
        ]
    ]
    .sort_values(
        "delivery_difference_days",
        ascending=False
    )
    .head(20)
)

In [ ]:
orders["delivered_late"] = (
    orders["delivery_difference_days"] > 0
)

In [ ]:
orders["delivered_late"] = pd.NA

delivered_mask = orders["order_status"] == "delivered"

orders.loc[delivered_mask, "delivered_late"] = (
    orders.loc[delivered_mask, "delivery_difference_days"] > 0
)

In [ ]:
print(orders["delivered_late"].value_counts(dropna=False))

In [ ]:
late_rate = (
    orders.loc[delivered_mask, "delivered_late"]
    .mean()
    * 100
)

print(f"Late delivery rate: {late_rate:.2f}%")

In [ ]:
print(
    orders[
        (orders["order_status"] == "delivered") &
        (orders["delivery_difference_days"] > 0)
    ][
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_difference_days",
            "delivered_late"
        ]
    ].head(10)
)

In [ ]:
print(
    orders[
        orders["delivery_difference_days"].notna()
    ]["delivery_difference_days"].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
orders["delivery_status"] = pd.NA

orders.loc[
    orders["delivery_difference_days"] <= 0,
    "delivery_status"
] = "Early / On Time"

orders.loc[
    orders["delivery_difference_days"] > 0,
    "delivery_status"
] = "Late"

In [ ]:
print(
    orders["delivery_status"].value_counts(dropna=False)
)

In [ ]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
]

late_delivery_rate = (
    delivered_orders["delivery_status"].eq("Late").mean() * 100
)

print(f"Late delivery rate: {late_delivery_rate:.2f}%")

In [ ]:
print(orders.isna().sum())

In [ ]:
print("Duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

In [ ]:
print(
    orders[
        orders["delivery_days"].isna() &
        orders["order_delivered_customer_date"].notna()
    ][
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ]
)

In [ ]:
print(
    orders[
        orders["delivery_status"].isna() &
        orders["delivered_late"].notna()
    ][
        [
            "order_id",
            "order_status",
            "delivery_status",
            "delivered_late"
        ]
    ]
)

In [ ]:
orders.loc[
    orders["delivery_difference_days"].isna(),
    "delivered_late"
] = pd.NA

In [ ]:
print(
    orders[
        orders["delivery_status"].isna() &
        orders["delivered_late"].notna()
    ]
)

In [ ]:
extreme_delivery = orders[
    orders["delivery_days"] > 100
]

print(
    extreme_delivery[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "delivery_difference_days"
        ]
    ].sort_values(
        "delivery_days",
        ascending=False
    )
)

In [ ]:
print(
    "Orders with delivery > 100 days:",
    orders["delivery_days"].gt(100).sum()
)

print(
    "Orders with delivery > 30 days:",
    orders["delivery_days"].gt(30).sum()
)

print(
    "Orders with delivery > 60 days:",
    orders["delivery_days"].gt(60).sum()
)

print(
    "Orders with delivery > 90 days:",
    orders["delivery_days"].gt(90).sum()
)

In [ ]:
print(
    orders["order_delivered_customer_date"]
    .value_counts()
    .head(20)
)

In [ ]:
print(
    orders[
        orders["order_delivered_customer_date"]
        .dt.strftime("%Y-%m-%d")
        .isin(["2017-09-19", "2018-09-19"])
    ][
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days"
        ]
    ].sort_values("order_delivered_customer_date")
)

In [ ]:
print(
    extreme_delivery[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "delivery_days"
        ]
    ].head(20)
)

In [ ]:
print(
    extreme_delivery[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days",
            "delivery_difference_days"
        ]
    ].sort_values("delivery_days", ascending=False).head(20)
)

In [ ]:
print(
    extreme_delivery["order_delivered_customer_date"]
    .dt.date
    .value_counts()
    .head(20)
)

In [ ]:
extreme_delivery[
    "order_delivered_customer_date"
].dt.date.value_counts()

In [ ]:
problem_date_orders = orders[
    orders["order_delivered_customer_date"].dt.date == pd.Timestamp("2017-09-19").date()
]

print(
    problem_date_orders[
        [
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_days"
        ]
    ].sort_values("order_purchase_timestamp")
)

In [ ]:
print(
    "Negative delivery days:",
    (orders["delivery_days"] < 0).sum()
)

print(
    "Delivery before carrier:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum()
)

print(
    "Duplicate order IDs:",
    orders["order_id"].duplicated().sum()
)

In [ ]:
print(orders.isna().sum())

In [ ]:
orders = orders.drop(columns=["delivered_late"])

In [ ]:
print(orders.columns)

In [ ]:
print(orders.info())

In [ ]:
print(order_items.info())

In [ ]:
print(order_items.describe())

In [ ]:
print(order_items.head())

In [ ]:
print(order_items.isna().sum())

In [ ]:
print("Duplicate rows:", order_items.duplicated().sum())

print(
    "Duplicate order-item combinations:",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

In [ ]:
print(
    "Unknown order IDs:",
    (~order_items["order_id"].isin(orders["order_id"])).sum()
)

print(
    "Unknown product IDs:",
    (~order_items["product_id"].isin(products["product_id"])).sum()
)

print(
    "Unknown seller IDs:",
    (~order_items["seller_id"].isin(sellers["seller_id"])).sum()
)

In [ ]:
print("Price <= 0:", (order_items["price"] <= 0).sum())

print("Freight < 0:", (order_items["freight_value"] < 0).sum())

In [ ]:
print("Current order_items rows:", len(order_items))
print("Expected original rows:", 112650)
print("Difference:", 112650 - len(order_items))

In [ ]:
print(
    order_items["shipping_limit_date"]
    .describe()
)

In [ ]:
print(
    order_items["shipping_limit_date"]
    .dt.year.value_counts()
    .sort_index()
)

In [ ]:
order_items["item_total_value"] = (
    order_items["price"] + order_items["freight_value"]
)

order_items["freight_percentage"] = (
    order_items["freight_value"] / order_items["price"]
) * 100

print(
    order_items[
        [
            "price",
            "freight_value",
            "item_total_value",
            "freight_percentage"
        ]
    ].head()
)

In [ ]:
print(
    order_items[
        ["item_total_value", "freight_percentage"]
    ].describe()
)

In [ ]:
print("Missing values:")
print(order_items.isna().sum())

print("\nDuplicate rows:", order_items.duplicated().sum())

# PAYMENTS — INITIAL INSPECTION

In [ ]:
print(payments.info())

In [ ]:
print(payments.describe())

In [ ]:
print(payments.head())

In [ ]:
print("\nMissing values:")
print(payments.isna().sum())

In [ ]:
print("\nDuplicate rows:")
print(payments.duplicated().sum())

# ==========================================
# PAYMENTS — BASIC VALIDATION
# ==========================================

In [ ]:
print(
    "Unknown order IDs:",
    (~payments["order_id"].isin(orders["order_id"])).sum()
)

print(
    "Payment value <= 0:",
    (payments["payment_value"] <= 0).sum()
)

print(
    "Payment installments < 1:",
    (payments["payment_installments"] < 1).sum()
)

print(
    "Payment type counts:"
)

print(
    payments["payment_type"].value_counts(dropna=False)
)

# ==========================================
# PAYMENTS — ORDER/PAYMENT STRUCTURE
# ==========================================

In [ ]:
print(
    "Unique orders:",
    payments["order_id"].nunique()
)

print(
    "Total payment records:",
    len(payments)
)

print(
    "Orders with multiple payment records:",
    (payments.groupby("order_id").size() > 1).sum()
)

In [ ]:
print(
    payments.groupby("order_id")["payment_value"]
    .sum()
    .describe()
)

In [ ]:
print(
    payments.groupby("order_id")["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
print(
    payments[payments["payment_value"].isna()]
    ["payment_type"]
    .value_counts(dropna=False)
)

In [ ]:
print(
    payments[payments["payment_value"].isna()]
    ["payment_installments"]
    .value_counts(dropna=False)
)

In [ ]:
print(
    payments[payments["payment_value"].isna()]
    .head(20)
)

In [ ]:
missing_payment_orders = payments[
    payments["payment_value"].isna()
]["order_id"]

print("Missing payment rows:", len(missing_payment_orders))
print("Unique orders affected:", missing_payment_orders.nunique())

print(
    "Orders with missing payment_value and other payment records:",
    payments[
        payments["order_id"].isin(missing_payment_orders)
    ]
    .groupby("order_id")
    .size()
    .gt(1)
    .sum()
)

In [ ]:
# Keep only payment records with a known payment value
payments_clean = payments[
    payments["payment_value"].notna()
].copy()

print("Original payment rows:", len(payments))
print("Clean payment rows:", len(payments_clean))
print("Rows removed:", len(payments) - len(payments_clean))

print("\nMissing payment values:")
print(payments_clean["payment_value"].isna().sum())

In [ ]:
print(
    payments_clean[
        payments_clean["payment_value"] <= 0
    ]
)

In [ ]:
print(
    payments_clean[
        payments_clean["payment_type"] == "not_defined"
    ]
)

In [ ]:
print(
    payments_clean["payment_type"].value_counts()
)

In [ ]:
payments_clean = payments_clean[
    payments_clean["payment_type"] != "not_defined"
].copy()

print("Payment rows after cleaning:", len(payments_clean))
print("\nPayment types:")
print(payments_clean["payment_type"].value_counts())

In [ ]:
print("Missing payment values:", payments_clean["payment_value"].isna().sum())
print("Invalid payment values:", (payments_clean["payment_value"] < 0).sum())
print("Not defined payments:", (payments_clean["payment_type"] == "not_defined").sum())
print("Duplicate rows:", payments_clean.duplicated().sum())

In [ ]:
payment_summary = (
    payments_clean
    .groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count")
    )
)

print(payment_summary.head())
print(payment_summary.info())

In [ ]:
print("Payment summary rows:", len(payment_summary))

print(
    "Duplicate order IDs:",
    payment_summary["order_id"].duplicated().sum()
)

print(
    "Unknown order IDs:",
    (~payment_summary["order_id"].isin(orders["order_id"])).sum()
)

print(
    "Missing total payment:",
    payment_summary["total_payment_value"].isna().sum()
)

print(
    "Invalid total payment:",
    (payment_summary["total_payment_value"] < 0).sum()
)

In [ ]:
print(
    payment_summary["total_payment_value"].describe()
)

In [ ]:
print(
    payment_summary["payment_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
print(
    payment_summary["payment_count"]
    .value_counts()
    .sort_index()
)

In [ ]:
print(
    payment_summary[
        payment_summary["payment_count"] > 1
    ]
    .sort_values("payment_count", ascending=False)
    .head(10)
)

# reviews

In [ ]:
print(reviews.info())

In [ ]:
print(reviews.head())

In [ ]:
print(reviews.describe())

In [ ]:
print("\nMissing values:")
print(reviews.isna().sum())

In [ ]:
print("\nDuplicate rows:")
print(reviews.duplicated().sum())

In [ ]:
print(
    "Duplicate review IDs:",
    reviews["review_id"].duplicated().sum()
)

In [ ]:
print(
    "Unknown order IDs:",
    (~reviews["order_id"].isin(orders["order_id"])).sum()
)

In [ ]:
print(
    "Invalid review scores:",
    (
        (reviews["review_score"] < 1) |
        (reviews["review_score"] > 5)
    ).sum()
)

In [ ]:
print(
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

In [ ]:
print(
    "Orders with multiple reviews:",
    (reviews.groupby("order_id").size() > 1).sum()
)

In [ ]:
print(
    reviews[reviews["review_id"].duplicated(keep=False)]
    .sort_values("review_id")
    .head(30)
)

In [ ]:
duplicate_reviews = reviews[
    reviews["review_id"].duplicated(keep=False)
]

print(
    "Duplicate review IDs:",
    duplicate_reviews["review_id"].nunique()
)

print(
    "Duplicate review IDs linked to multiple orders:",
    (
        duplicate_reviews
        .groupby("review_id")["order_id"]
        .nunique()
        .gt(1)
        .sum()
    )
)

In [ ]:
print(
    "Exact duplicates among duplicate review IDs:",
    duplicate_reviews.duplicated().sum()
)

In [ ]:
print(
    reviews.groupby("order_id")["review_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

In [ ]:
reviews["review_score"].value_counts().sort_index()

In [ ]:
reviews["rating_category"] = pd.cut(
    reviews["review_score"],
    bins=[0, 2, 3, 4, 5],
    labels=["Negative", "Neutral", "Good", "Excellent"]
)

In [ ]:
print(reviews["rating_category"].value_counts().sort_index())

In [ ]:
reviews["review_response_days"] = (
    reviews["review_answer_timestamp"]
    - reviews["review_creation_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [ ]:
print(reviews["review_response_days"].describe())

In [ ]:
print(
    "Negative response days:",
    (reviews["review_response_days"] < 0).sum()
)

In [ ]:
print(
    reviews[
        ["review_score", "review_response_days"]
    ].describe()
)

In [ ]:
print(
    "Missing values after cleaning:"
)
print(reviews.isna().sum())

# products

In [ ]:
print(products.info())

In [ ]:
print(products.describe())

In [ ]:
print(products.head())

In [ ]:
print("\nMissing values:")
print(products.isna().sum())

In [ ]:
print("\nDuplicate rows:")
print(products.duplicated().sum())

In [ ]:
print(products[products.isna().any(axis=1)].head(20))

In [ ]:
print(
    products[
        ["product_name_length",
         "product_description_length",
         "product_photos_qty",
         "product_weight_g",
         "product_length_cm",
         "product_height_cm",
         "product_width_cm"]
    ].isna().sum(axis=1).value_counts().sort_index()
)

In [ ]:
missing_cols = products.isna().groupby(
    ["product_name_length",
     "product_description_length",
     "product_photos_qty",
     "product_weight_g",
     "product_length_cm",
     "product_height_cm",
     "product_width_cm"]
).size()

print(missing_cols)

In [ ]:
products_clean = products.dropna(
    subset=[
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
).copy()

print(products_clean.shape)

In [ ]:
print(products_clean["product_category_name_english"].isna().sum())

print(
    products_clean.loc[
        products_clean["product_category_name_english"].isna(),
        "product_category_name"
    ].value_counts().head(20)
)

In [ ]:
products_clean["product_category_name_english"] = (
    products_clean["product_category_name_english"]
    .fillna("unknown")
)

print(products_clean["product_category_name_english"].isna().sum())

In [ ]:
products_clean = products_clean.drop(columns=["product_weight_g"])

print(products_clean.shape)

In [ ]:
print("\nMissing values:")
print(products_clean.isna().sum())

print("\nDuplicate rows:", products_clean.duplicated().sum())
print("Duplicate product IDs:", products_clean["product_id"].duplicated().sum())

# sellers

In [ ]:
print(sellers.info())

In [ ]:
print(sellers.describe())

In [ ]:
print("Duplicate rows:", sellers.duplicated().sum())
print("Duplicate seller IDs:", sellers["seller_id"].duplicated().sum())

print("\nInvalid ZIP prefixes:")
print((sellers["seller_zip_code_prefix"] <= 0).sum())

sellers_clean = sellers.drop(columns=["seller_zip_code_prefix"]).copy()

print(sellers_clean.head())
print(sellers_clean.shape)

# Geolocation — Data Cleaning & Validation

In [ ]:
print(geolocation.info())

In [ ]:
print("Duplicate rows:", geolocation.duplicated().sum())

print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)

print(
    "Duplicate ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].duplicated().sum()
)

In [ ]:
zip_summary = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        coordinate_count=("geolocation_lat", "count"),
        unique_lat=("geolocation_lat", "nunique"),
        unique_lng=("geolocation_lng", "nunique")
    )
)

print(zip_summary.describe())

In [ ]:
geolocation_clean = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", "first"),
        geolocation_state=("geolocation_state", "first")
    )
)

print(geolocation_clean.shape)
print(geolocation_clean.head())

In [ ]:
geolocation_clean = geolocation_clean.drop(
    columns=["geolocation_lat", "geolocation_lng"]
).copy()

print(geolocation_clean.shape)
print(geolocation_clean.head())

In [ ]:
print("Missing values:")
print(geolocation_clean.isna().sum())

print("\nDuplicate rows:", geolocation_clean.duplicated().sum())

print(
    "Duplicate ZIP prefixes:",
    geolocation_clean["geolocation_zip_code_prefix"].duplicated().sum()
)

# Category Translation|

In [ ]:
print(category_translation.info())

In [ ]:
print("Duplicate rows:", category_translation.duplicated().sum())

print(
    "Duplicate Portuguese categories:",
    category_translation["product_category_name"].duplicated().sum()
)

In [ ]:
%whos

In [ ]:
customers_clean = customers.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
sellers_clean = sellers.drop(
    columns=["seller_zip_code_prefix"],
    errors="ignore"
).copy()

In [ ]:
print("Customers:", customers_clean.shape)
print("Orders:", orders_clean.shape)
print("Order Items:", order_items_clean.shape)
print("Payments:", payments_clean.shape)
print("Reviews:", reviews.shape)
print("Products:", products_clean.shape)
print("Sellers:", sellers_clean.shape)
print("Geolocation:", geolocation_clean.shape)
print("Category Translation:", category_translation.shape)

In [ ]:
import os

os.makedirs("../cleaned_data", exist_ok=True)

cleaned_datasets = {
    "customers_clean": customers_clean,
    "orders_clean": orders_clean,
    "order_items_clean": order_items_clean,
    "payments_clean": payments_clean,
    "reviews_clean": reviews,
    "products_clean": products_clean,
    "sellers_clean": sellers_clean,
    "geolocation_clean": geolocation_clean,
    "category_translation_clean": category_translation,
    "payment_summary": payment_summary,
}

for name, data in cleaned_datasets.items():
    path = f"../cleaned_data/{name}.csv"
    data.to_csv(path, index=False)
    print(f"Saved {name}: {data.shape}")